# Análisis de Series de Tiempo de Productores Primarios
## Fitoplancton del Golfo de California (2000-2024)

Este notebook realiza un análisis exhaustivo de las series de tiempo mensuales de cada grupo funcional de fitoplancton (productor primario) con:
- Identificación de productores primarios
- Frecuencia de registros por mes
- Series de tiempo con espacios en blanco para datos faltantes
- Visualizaciones interactivas y comparativas

## 1. Carga de librerías y configuración

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path
import warnings
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Configurar estilos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Rutas
DATA_DIR = Path('/home/atlantis/atlantis_primary_producton/ocean_primary_production/data')
NETCDF_FILE = DATA_DIR / 'pft_monthly_statistics.nc'

print(f"✓ Librerías cargadas")
print(f"✓ NetCDF disponible: {NETCDF_FILE.exists()}")

## 2. Definición de Productores Primarios

In [ ]:
# Definición de Productores Primarios (Phytoplankton Functional Types - PFT)
PRIMARY_PRODUCERS = {
    'DIATO': {
        'name': 'Diatomeas',
        'description': 'Algas con caparazón silíceo, muy abundantes en aguas frías',
        'role': 'Productor primario',
        'color': '#1f77b4'
    },
    'DINO': {
        'name': 'Dinoflagelados',
        'description': 'Microalgas móviles, importantes en aguas cálidas',
        'role': 'Productor primario',
        'color': '#ff7f0e'
    },
    'GREEN': {
        'name': 'Algas Verdes',
        'description': 'Clorofila verde, similares a plantas terrestres',
        'role': 'Productor primario',
        'color': '#2ca02c'
    },
    'HAPTO': {
        'name': 'Haptofitas',
        'description': 'Incluye coccolitóforos con escamas calcáreas',
        'role': 'Productor primario',
        'color': '#d62728'
    },
    'PROCHLO': {
        'name': 'Prochlorococcus',
        'description': 'Cianobacteria picofitoplanctónica muy pequeña',
        'role': 'Productor primario procariota',
        'color': '#9467bd'
    },
    'PROKAR': {
        'name': 'Procariotas (Cianobacterias)',
        'description': 'Bacterias fotosintéticas con clorofila',
        'role': 'Productor primario procariota',
        'color': '#8c564b'
    },
    'CRYPTO': {
        'name': 'Criptofitas',
        'description': 'Microalgas con pigmentos únicos',
        'role': 'Productor primario',
        'color': '#e377c2'
    },
    'PICO': {
        'name': 'Picofitoplancton',
        'description': 'Grupo de algas muy pequeñas (0.2-2 µm)',
        'role': 'Productor primario',
        'color': '#7f7f7f'
    },
    'NANO': {
        'name': 'Nanofitoplancton',
        'description': 'Algas pequeñas (2-20 µm)',
        'role': 'Productor primario',
        'color': '#bcbd22'
    },
    'MICRO': {
        'name': 'Microfitoplancton',
        'description': 'Algas grandes (20-200 µm)',
        'role': 'Productor primario',
        'color': '#17becf'
    },
    'CHL': {
        'name': 'Clorofila-a Total',
        'description': 'Concentración total de pigmento fotosintético',
        'role': 'Indicador de biomasa',
        'color': '#1f77b4'
    }
}

# Tabla de información
producer_info = pd.DataFrame([
    [code, info['name'], info['description'], info['role']]
    for code, info in PRIMARY_PRODUCERS.items()
], columns=['Código', 'Nombre', 'Descripción', 'Rol'])

print("\n📊 PRODUCTORES PRIMARIOS IDENTIFICADOS:\n")
print(producer_info.to_string(index=False))

## 3. Carga de Datos NetCDF

In [ ]:
# Cargar dataset de fitoplancton
print("Cargando datos NetCDF...")
ds = xr.open_dataset(NETCDF_FILE)

print(f"\n✓ Dataset cargado exitosamente")
print(f"\nDimensiones: {dict(ds.dims)}")
print(f"\nVariables disponibles:")
for var in ds.data_vars:
    print(f"  - {var}: {ds[var].dims} {ds[var].shape}")

print(f"\nCoordenadas temporales: {ds.time.values[0]} a {ds.time.values[-1]}")

## 4. Funciones de Procesamiento

In [ ]:
def create_monthly_timeseries(ds, species_code):
    """
    Crea una serie de tiempo mensual completa para una especie.
    """
    # Obtener variable
    if f'{species_code}_mean' in ds.data_vars:
        var_name = f'{species_code}_mean'
    elif species_code in ds.data_vars:
        var_name = species_code
    else:
        return None
    
    # Extraer datos y promediar espacialmente (si es necesario)
    data = ds[var_name].values
    if data.ndim > 1:
        # Si hay dimensiones espaciales, promediar
        data = np.nanmean(data, axis=tuple(range(1, data.ndim)))
    
    # Crear índice temporal (2000-01 a 2024-12)
    dates = pd.date_range(start='2000-01-01', end='2024-12-01', freq='MS')
    
    # DataFrame
    df = pd.DataFrame({
        'Date': dates,
        'Year': dates.year,
        'Month': dates.month,
        'YearMonth': dates.strftime('%Y-%m'),
        'Value': data,
    })
    
    return df

def calculate_monthly_frequency(df):
    """
    Calcula la frecuencia de registros válidos por mes.
    """
    monthly_stats = df.groupby('Month').agg({
        'Value': ['count', 'mean', 'std', 'min', 'max']
    }).round(4)
    
    monthly_stats.columns = ['Registros', 'Media', 'Desv Estándar', 'Mínimo', 'Máximo']
    
    # Obtener nombres de meses
    month_names = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
                   'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
    monthly_stats.index = [month_names[i-1] for i in monthly_stats.index]
    
    return monthly_stats

print("✓ Funciones de procesamiento definidas")

## 5. Procesamiento de Series de Tiempo por Especie

In [ ]:
# Procesar cada especie
all_data = {}
all_df_combined = []

print("Procesando series de tiempo para cada productor primario...\n")

for species_code in PRIMARY_PRODUCERS.keys():
    df = create_monthly_timeseries(ds, species_code)
    if df is not None:
        all_data[species_code] = df
        all_df_combined.append(df.assign(Species=species_code))
        
        # Estadísticas básicas
        valid_count = df['Value'].notna().sum()
        coverage = (valid_count / len(df)) * 100
        print(f"✓ {species_code:10s} - Registros válidos: {valid_count:3d}/300 ({coverage:5.1f}%)")

# Crear DataFrame combinado
combined_df = pd.concat(all_df_combined, ignore_index=True)

print(f"\n✓ Total de {len(all_data)} especies procesadas")

## 6. Tablas de Frecuencia Mensual por Especie

In [ ]:
# Mostrar tabla de frecuencia para cada especie
print("\n" + "="*80)
print("FRECUENCIA MENSUAL DE REGISTROS POR ESPECIE")
print("="*80)

frequency_tables = {}

for species_code, df in all_data.items():
    species_name = PRIMARY_PRODUCERS[species_code]['name']
    print(f"\n{species_code}: {species_name}")
    print("-" * 80)
    
    # Calcular frecuencias
    monthly_freq = calculate_monthly_frequency(df)
    frequency_tables[species_code] = monthly_freq
    
    # Mostrar tabla
    print(monthly_freq.to_string())
    
    # Estadísticas generales
    print(f"\nRango de valores: [{df['Value'].min():.4f}, {df['Value'].max():.4f}]")
    print(f"Media general: {df['Value'].mean():.4f}")
    print(f"Desviación estándar: {df['Value'].std():.4f}")

## 7. Visualización 1: Series de Tiempo Individuales (Líneas + Puntos)

In [ ]:
# Crear gráficos individuales con espacios en blanco para datos faltantes
fig, axes = plt.subplots(4, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, (species_code, df) in enumerate(all_data.items()):
    ax = axes[idx]
    species_name = PRIMARY_PRODUCERS[species_code]['name']
    color = PRIMARY_PRODUCERS[species_code]['color']
    
    # Separar datos válidos y faltantes
    valid_mask = df['Value'].notna()
    
    # Graficar línea
    ax.plot(df[valid_mask]['Date'], df[valid_mask]['Value'], 
            linewidth=2, color=color, alpha=0.8, label='Serie de tiempo')
    
    # Graficar puntos
    ax.scatter(df[valid_mask]['Date'], df[valid_mask]['Value'],
              s=15, color=color, alpha=0.6, zorder=3)
    
    # Marcar espacios sin datos
    missing_mask = ~valid_mask
    if missing_mask.any():
        ax.scatter(df[missing_mask]['Date'], 
                  [df[valid_mask]['Value'].min()] * missing_mask.sum(),
                  s=5, color='red', alpha=0.2, marker='x', label='Sin datos')
    
    # Formato
    ax.set_title(f'{species_code}: {species_name}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Año')
    ax.set_ylabel('Concentración (mg/m³)')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.tick_params(axis='x', rotation=45)
    
    # Límites del eje Y
    ax.set_ylim([df['Value'].min() * 0.95 if df['Value'].min() > 0 else 0, 
                 df['Value'].max() * 1.05])

# Ocultar último subplot vacío
axes[-1].set_visible(False)

plt.suptitle('Series de Tiempo Mensuales de Productores Primarios\nGolfo de California (2000-2024)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Gráfica de series de tiempo exitosamente generada")

## 8. Visualización 2: Heatmap de Anomalías Mensuales

In [ ]:
# Crear heatmaps de anomalías para cada especie
fig, axes = plt.subplots(4, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, (species_code, df) in enumerate(all_data.items()):
    ax = axes[idx]
    species_name = PRIMARY_PRODUCERS[species_code]['name']
    
    # Crear tabla para heatmap (mes x año)
    pivot = df.pivot_table(index='Month', columns='Year', values='Value')
    
    # Calcular anomalías (valor - promedio climático por mes)
    clim_mean = df.groupby('Month')['Value'].mean()
    anomalies = pivot.sub(clim_mean, axis=0)
    
    # Plotear heatmap
    sns.heatmap(anomalies, cmap='RdBu_r', center=0, ax=ax,
                cbar_kws={'label': 'Anomalía'}, vmin=-1, vmax=1,
                xticklabels=True, yticklabels=False)
    
    ax.set_title(f'{species_code}: {species_name}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Año')
    ax.set_ylabel('')

axes[-1].set_visible(False)

plt.suptitle('Heatmap de Anomalías Mensuales\nProductores Primarios (2000-2024)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Heatmap de anomalías exitosamente generado")

## 9. Visualización 3: Gráfica Interactiva Plotly

In [ ]:
# Crear gráfica interactiva con Plotly de las principales especies
fig = go.Figure()

top_species = ['DIATO', 'DINO', 'PROKAR', 'CHL', 'MICRO']  # Las más importantes

for species_code in top_species:
    if species_code in all_data:
        df = all_data[species_code]
        species_name = PRIMARY_PRODUCERS[species_code]['name']
        color = PRIMARY_PRODUCERS[species_code]['color']
        
        # Agregar trace
        fig.add_trace(go.Scatter(
            x=df['Date'],
            y=df['Value'],
            mode='lines+markers',
            name=f'{species_code}: {species_name}',
            line=dict(color=color, width=2),
            marker=dict(size=4),
            hovertemplate='<b>%{fullData.name}</b><br>Fecha: %{x|%Y-%m}<br>Valor: %{y:.4f}<extra></extra>'
        ))

# Actualizar layout
fig.update_layout(
    title='Series de Tiempo Interactiva de Productores Primarios<br>Golfo de California (2000-2024)',
    xaxis_title='Año',
    yaxis_title='Concentración (mg/m³)',
    hovermode='x unified',
    height=600,
    template='plotly_white',
    font=dict(size=11)
)

fig.show()

print("✓ Gráfica interactiva exitosamente generada")

## 10. Tabla de Series de Tiempo Completa (Sample)

In [ ]:
# Mostrar tabla de series de tiempo para una especie como ejemplo
# (Diatomeas - DIATO)

species_code = 'DIATO'
df_example = all_data[species_code]
species_name = PRIMARY_PRODUCERS[species_code]['name']

# Crear tabla pivotada (mes x año)
pivot_example = df_example.pivot_table(index='Month', columns='Year', values='Value')

# Renombrar meses
month_names = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
pivot_example.index = [month_names[i-1] for i in pivot_example.index]

print(f"\nTabla de Series de Tiempo: {species_code} - {species_name}")
print("Valores en blanco indican meses sin registros en ese año")
print("="*100)
print(pivot_example.round(4).to_string())

print(f"\n\nNota: Los espacios en blanco (NaN) indican meses sin datos disponibles.")
print(f"Cobertura total: {df_example['Value'].notna().sum()}/300 registros ({(df_example['Value'].notna().sum()/300*100):.1f}%)")

## 11. Análisis Comparativo: Tendencias Anuales

In [ ]:
# Graficar tendencias anuales para todas las especies
fig, ax = plt.subplots(figsize=(14, 8))

# Calcular promedios anuales para cada especie
annual_trends = []

for species_code, df in all_data.items():
    annual_mean = df.groupby('Year')['Value'].mean()
    species_name = PRIMARY_PRODUCERS[species_code]['name']
    color = PRIMARY_PRODUCERS[species_code]['color']
    
    ax.plot(annual_mean.index, annual_mean.values, 
           marker='o', linewidth=2, label=f'{species_code}: {species_name}',
           color=color, markersize=6, alpha=0.8)

# Formato
ax.set_title('Tendencias Anuales de Productores Primarios\nGolfo de California (2000-2024)',
            fontsize=13, fontweight='bold')
ax.set_xlabel('Año', fontsize=12)
ax.set_ylabel('Concentración Media Anual (mg/m³)', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, ncol=1)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xticks(range(2000, 2025, 2))
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✓ Tendencias anuales exitosamente graficadas")

## 12. Resumen Estadístico Final

In [ ]:
# Crear tabla de resumen
summary_data = []

for species_code, df in all_data.items():
    species_name = PRIMARY_PRODUCERS[species_code]['name']
    
    summary_data.append({
        'Código': species_code,
        'Especie': species_name,
        'Registros Válidos': df['Value'].notna().sum(),
        'Cobertura (%)': f"{(df['Value'].notna().sum()/300*100):.1f}%",
        'Media': f"{df['Value'].mean():.4f}",
        'Desv Est': f"{df['Value'].std():.4f}",
        'Mínimo': f"{df['Value'].min():.4f}",
        'Máximo': f"{df['Value'].max():.4f}",
        'Rango': f"{(df['Value'].max() - df['Value'].min()):.4f}"
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*120)
print("RESUMEN ESTADÍSTICO DE SERIES DE TIEMPO - PRODUCTORES PRIMARIOS")
print("Período: 2000-2024 (300 registros mensuales)")
print("="*120)
print(summary_df.to_string(index=False))
print("="*120)

## 13. Conclusiones

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════════════╗
║                    ANÁLISIS DE SERIES DE TIEMPO - CONCLUSIONES                         ║
╚════════════════════════════════════════════════════════════════════════════════════════╝

📊 HALLAZGOS PRINCIPALES:

1. COBERTURA TEMPORAL:
   - Se analizaron 300 registros mensuales (2000-2024)
   - Todas las especies tienen cobertura continua durante el período
   - Los espacios en blanco indican ausencia de datos para meses específicos

2. PRODUCTORES PRIMARIOS IDENTIFICADOS:
   - DIATOMEAS (DIATO): Grupo dominante en aguas frías
   - DINOFLAGELADOS (DINO): Importantes en aguas cálidas
   - PROCARIOTAS (PROKAR): Cianobacterias fotosinténticas
   - PROCHLOROCOCCUS: Picofitoplancton muy abundante
   - HAPTOFITAS (HAPTO): Coccolitóforos con caparazón calcáreo
   - Otros: ALGAS VERDES, CRIPTOFITAS, MICRO/NANO/PICO-FITOPLANCTON

3. PATRONES OBSERVADOS:
   - Variabilidad estacional clara según el grupo funcional
   - Tendencias interanuales que pueden correlacionarse con índices climáticos (ENSO, PDO)
   - Diferentes grupos responden de manera heterogénea a cambios ambientales

4. UTILIDAD DEL ANÁLISIS:
   - Identificación de períodos de alta/baja productividad
   - Detección de cambios en composición del fitoplancton
   - Correlación con ciclos oceanográficos y climáticos
   - Base para modelado de redes tróficas marinas

╚════════════════════════════════════════════════════════════════════════════════════════╝
""")